# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their Croissant `@id` fields throughout.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and discover available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Dataset object
dataset = mlc.Dataset(croissant_url)

# Print the main metadata fields
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")

## 2. Data Overview
Explore available record sets and their `@id`s, fields, and columns. We will use these IDs for all programmatic referencing and data manipulation below.

In [ ]:
# List all available record sets with their @id fields

print("Available record sets (referenced by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', None)}")
    # List available fields within this record set
    print("  Fields:")
    for f in rs.fields:
        print(f"    - @id: {f.id} | name: {getattr(f, 'name', None)}")
    # List available columns in the record set, if present
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for c in rs.columns:
            print(f"    - @id: {c.id} | name: {getattr(c, 'name', None)}")
    print()

## 3. Data Extraction
Select specific record sets to load. For all data access, we will use record set and field `@id`s exactly as listed above.

In [ ]:
# Gather all available record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load all record sets into Pandas DataFrames, keyed by their @id
dataframes = {}
for record_set_id in record_set_ids:
    # Use the mlcroissant API for streaming records with reference by @id
    records_list = list(dataset.records(record_set=record_set_id))
    if records_list:
        dataframes[record_set_id] = pd.DataFrame(records_list)
    else:
        print(f"[Info] No records found for record set @id: {record_set_id}")

# Print the columns (field @ids) of the first non-empty record set
if dataframes:
    first_record_set = list(dataframes.keys())[0]
    print(f"First loaded record set @id: {first_record_set}")
    print("Field @ids in this record set:")
    print(list(dataframes[first_record_set].columns))
    display(dataframes[first_record_set].head())
else:
    print("No dataframes available after record extraction.")

## 4. Exploratory Data Analysis (EDA)
We will perform EDA on one of the populated record sets using its field `@id` for numeric/categorical processing. Please adjust or rerun with your preferred field and record set as needed.

In [ ]:
# Identify a DataFrame and numeric field @id for EDA
if dataframes:
    # Pick the first available DataFrame
    recset_id = list(dataframes.keys())[0]
    df = dataframes[recset_id]
    # Find a numeric column by inspecting data type
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Using field @id '{numeric_field_id}' for EDA in record set @id '{recset_id}'")
        # Filtering example: filter values greater than arbitrary threshold
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another (possibly categorical) field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            print(f"Grouped (mean of numeric fields) by group_field @id '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(grouped_df.head())
        else:
            print("No suitable group-by field available.")
    else:
        print("No numeric field could be inferred in the selected record set.")
else:
    print("No data available for EDA. Please check previous steps.")

## 5. Visualization
Visualize the distribution of a selected numeric field and its relation to a grouping variable, referencing fields by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure EDA variables from previous cell
if dataframes and 'numeric_field_id' in locals() and numeric_field_id is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id + " (@id)")
    plt.ylabel("Frequency")
    plt.show()

    # If group_field exists, plot boxplots
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field} (@id)")
        plt.xlabel(group_field + " (@id)")
        plt.ylabel(numeric_field_id + " (@id)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization. Please check EDA section.")

## 6. Conclusion
In this notebook, we demonstrated step-by-step how to load, explore, and process the FAIR^2 rangeland management dataset with `mlcroissant`. We referenced all schema entities entirely by their stable `@id`, ensuring robust and reproducible programmatic access. The EDA and visualization workflow can be easily adapted to other record sets and fields by updating the selected `@id`s in the code. Continue your analysis with further transforms, modelling, or domain-specific data science as required.